# **FUENTE DE LOS DATOS**

https://www.kaggle.com/competitions/student-performance-prediction-machine-learning-challenge/data?utm_source=chatgpt.com

# **IMPORTACIÓN DE LIBRERÍAS**

In [ ]:
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_validate, KFold
from sklearn.feature_selection import mutual_info_regression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# **CARGA DEL DATASET**

In [ ]:
def cargaDataset(train_path: str, test_path: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Carga los conjuntos de datos de entrenamiento y prueba.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: DataFrame de train y DataFrame de test.
    """
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)
    return df_train, df_test

train_path = "train.csv"
test_path = "test.csv"
df, df_test = cargaDataset(train_path, test_path)

# **ANÁLISIS INICIAL**

In [ ]:
display(df.sample(10))

In [ ]:
df.info()
display(df.describe())

In [ ]:
def verificar_ausencia_valores(df: pd.DataFrame) -> None:
    """Informa los valores nulos por columna y los registros duplicados del DataFrame."""
    valores_faltantes = df.isnull().sum()
    if (valores_faltantes > 0).any():
        print("=== VALORES NULOS POR COLUMNA ===")
        display(valores_faltantes[valores_faltantes > 0])
    else:
        print("No hay valores faltantes")

    valores_duplicados = df.duplicated().sum()
    if valores_duplicados != 0:
        print("\n=== REGISTROS DUPLICADOS ===")
        print(f"Cantidad de registros completamente duplicados: {valores_duplicados}")
    else:
        print("No hay registros duplicados")

verificar_ausencia_valores(df)

# **CAMBIO DEL NOMBRE DE LOS PREDICTORES DEL DATASET**

In [ ]:
traduccion = {
    'id': 'id',
    'student_id': 'id_estudiante',
    'age': 'edad',
    'gender': 'genero',
    'study_hours_per_day': 'horas_estudio_por_dia',
    'sleep_hours': 'horas_sueno',
    'phone_usage_hours': 'horas_uso_telefono',
    'social_media_hours': 'horas_redes_sociales',
    'youtube_hours': 'horas_youtube',
    'gaming_hours': 'horas_videojuegos',
    'breaks_per_day': 'descansos_por_dia',
    'coffee_intake_mg': 'consumo_cafe_mg',
    'exercise_minutes': 'minutos_ejercicio',
    'assignments_completed': 'tareas_completadas',
    'attendance_percentage': 'porcentaje_asistencia',
    'stress_level': 'nivel_estres',
    'focus_score': 'puntaje_enfoque',
    'final_grade': 'calificacion_final',
    'productivity_score': 'puntaje_productividad'
}

def traducir_columnas(df_dataset: pd.DataFrame, diccionario: dict) -> pd.DataFrame:
    """Traduce el nombre de las columnas de un DataFrame al español.

    Args:
        df_dataset (pd.DataFrame): El DataFrame original con nombres en inglés.
        diccionario (dict): Correspondencia entre nombres originales y traducidos.
    Returns:
        pd.DataFrame: Un nuevo DataFrame con las columnas traducidas al español.
    """
    return df_dataset.rename(columns=diccionario)

df = traducir_columnas(df, traduccion)
df_test = traducir_columnas(df_test, traduccion)

print("Columnas del conjunto de entrenamiento:", df.columns.tolist())
print("Columnas del dataset de prueba:", df_test.columns.tolist())

# **SEPARACIÓN DE LOS DATOS EN TRAIN Y  TEST**

### División de Datos para Validación Local

Vamos a separar nuestro conjunto **train.csv** en datos de entrenamiento **train** y de validación **val** para poder evaluar el rendimiento de nuestro modelo de forma local.

In [ ]:
# Excluimos las columnas identificadoras, que no aportan información, y la variable objetivo
def dividir_datos(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    """Separa el DataFrame en predictores y objetivo, y luego en entrenamiento (80 %) y validación (20 %)."""
    features = [col for col in df.columns if col not in ['id', 'id_estudiante', 'puntaje_productividad']]
    target = 'puntaje_productividad'
    X = df[features]
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = dividir_datos(df)
print(f"Dimensiones de X_train (Entrenamiento):  {X_train.shape}")
print(f"Dimensiones de X_test (Validación local): {X_test.shape}")

### **Visualización de Correlaciones (Mapa de Calor)**

In [ ]:
def graficar_mapa_calor(X_train: pd.DataFrame, y_train: pd.Series = None) -> None:
    """Genera y muestra un mapa de calor de las correlaciones del dataset.

    Args:
        X_train (pd.DataFrame): DataFrame con las variables predictoras.
        y_train (pd.Series, opcional): Serie con la variable objetivo a incluir en el análisis.
    """
    df_analisis = X_train.copy()
    if y_train is not None:
        df_analisis[y_train.name] = y_train

    # Seleccionamos únicamente las columnas numéricas para la correlación
    df_numerico = df_analisis.select_dtypes(include=['int64', 'float64'])

    # Calculamos la matriz de correlación
    corr_matrix = df_numerico.corr()

    # Configuramos el lienzo de matplotlib
    plt.figure(figsize=(14, 10))

    # Dibujamos el mapa de calor
    sns.heatmap(
        corr_matrix,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        linewidths=0.5,
        square=True
    )

    plt.title("Mapa de Calor de Correlaciones (Conjunto de Entrenamiento)", fontsize=16)
    plt.tight_layout()
    plt.show()


# Ejecutamos la función con el conjunto de entrenamiento (X_train) junto a su variable objetivo (y_train)
graficar_mapa_calor(X_train, y_train)

### **Comparativa de Relaciones (Pearson, Spearman e Información Mutua)**

In [ ]:
def calcular_informacion_mutua(X_train: pd.DataFrame, y_train: pd.Series) -> pd.Series:
    """Calcula la información mutua para todas las características, codificando
    temporalmente las columnas categóricas como 'genero', y devuelve los resultados como Serie.
    """
    X_temp = X_train.copy()

    for col in X_temp.select_dtypes(include=['object', 'category']).columns:
        X_temp[col] = X_temp[col].astype('category').cat.codes

    mi_scores = mutual_info_regression(X_temp, y_train, random_state=42)
    return pd.Series(mi_scores, index=X_train.columns)

In [ ]:
def mostrar_tabla_correlaciones_completas(X_train: pd.DataFrame, y_train: pd.Series) -> None:
    """Muestra Pearson, Spearman e información mutua de cada predictor con la variable objetivo."""
    df_temp = X_train.select_dtypes(include=['int64', 'float64']).copy()
    df_temp[y_train.name] = y_train

    corr_pearson = df_temp.corr(method='pearson')[y_train.name].drop(y_train.name)
    corr_spearman = df_temp.corr(method='spearman')[y_train.name].drop(y_train.name)

    info_mutua = calcular_informacion_mutua(X_train, y_train)

    df_corr = pd.DataFrame({
        'Pearson': corr_pearson,
        'Spearman': corr_spearman,
        'Informacion_Mutua': info_mutua
    }).reset_index()

    df_corr.rename(columns={'index': 'Variable'}, inplace=True)
    df_corr = df_corr.sort_values(by='Informacion_Mutua', ascending=False).reset_index(drop=True)
    print(f"Tabla Comparativa de Relaciones con {y_train.name}")
    display(df_corr)


mostrar_tabla_correlaciones_completas(X_train, y_train)

### **Visualización de Distribuciones de las Variables Numéricas**

In [ ]:
def graficar_distribucion_variables(X_train: pd.DataFrame, y_train: pd.Series) -> None:
    """Grafica la distribución de todas las variables numéricas de X_train y la variable objetivo y_train."""
    # Unimos las características numéricas y el target
    df_temp = X_train.select_dtypes(include=['int64', 'float64']).copy()
    df_temp[y_train.name] = y_train

    columnas_numericas = df_temp.columns.tolist()
    num_cols = len(columnas_numericas)

    # Calculamos una cuadrícula rectangular óptima para los subplots
    grid_cols = 4
    grid_rows = math.ceil(num_cols / grid_cols)

    plt.figure(figsize=(20, 4 * grid_rows))

    for i, col in enumerate(columnas_numericas, 1):
        plt.subplot(grid_rows, grid_cols, i)
        # Usamos histplot para ver la distribución y la curva de densidad (KDE)
        sns.histplot(df_temp[col], kde=True, color='royalblue', bins=30)
        plt.title(f'Distribución de {col}', fontsize=12)
        plt.xlabel('')
        plt.ylabel('Frecuencia')

    plt.tight_layout()
    plt.show()

graficar_distribucion_variables(X_train, y_train)

In [ ]:
display(X_train.describe())

## **Conclusiones del análisis de variables**

Evalué la relación de cada predictor con puntaje_productividad usando Pearson, Spearman e información mutua. Los tres métodos coinciden en los resultados.

**Variables que conservo:** horas_estudio_por_dia, puntaje_enfoque, horas_sueno, horas_uso_telefono, nivel_estres y porcentaje_asistencia. La más importante es horas_estudio_por_dia (r = 0.73).

**Variables que elimino:** edad, genero, consumo_cafe_mg, tareas_completadas, horas_redes_sociales, horas_youtube, horas_videojuegos, descansos_por_dia, minutos_ejercicio y calificacion_final.

**Por qué las elimino:**

- Su correlación es prácticamente nula (|r| < 0.02) y su información mutua es cero o despreciable, así que descarto relaciones lineales, monótonas y no lineales.
- Como tres métodos distintos llegan a lo mismo, considero que el resultado es confiable.
- Mantenerlas solo agregaría ruido y aumentaría el riesgo de sobreajuste sin mejorar las predicciones.

Además, como Pearson y Spearman dan valores casi iguales, concluyo que las relaciones son aproximadamente lineales, por lo que usaré una regresión lineal como modelo base.

# **ENTRENAMIENTO DEL MODELO**

El escalado con `StandardScaler` se hace dentro del `Pipeline`, así se ajusta solo con los datos de entrenamiento de cada fold y no hay fuga de información hacia la validación.

In [ ]:
COLUMNAS_ID = ['id', 'id_estudiante']

VARIABLES_SELECCIONADAS = [
    'horas_estudio_por_dia',
    'puntaje_enfoque',
    'horas_sueno',
    'horas_uso_telefono',
    'nivel_estres',
    'porcentaje_asistencia'
]

OPCIONES = {
    '1': 'Entrenar con las variables seleccionadas',
    '2': 'Entrenar con todas las variables'
}


# ================= PREPARACIÓN =================
def preparar_datos(X_train: pd.DataFrame, X_test: pd.DataFrame,
                   variables: list = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Quita los identificadores y deja las variables pedidas; con todas, codifica las categóricas."""
    X_tr = X_train.drop(columns=COLUMNAS_ID, errors='ignore').copy()
    X_te = X_test.drop(columns=COLUMNAS_ID, errors='ignore').copy()

    if variables is not None:
        return X_tr[variables], X_te[variables]

    categoricas = X_tr.select_dtypes(include=['object', 'category']).columns.tolist()
    X_tr = pd.get_dummies(X_tr, columns=categoricas, drop_first=True, dtype=int)
    X_te = pd.get_dummies(X_te, columns=categoricas, drop_first=True, dtype=int)
    X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)

    return X_tr, X_te


def crear_pipeline() -> Pipeline:
    return Pipeline([
        ('escalador', StandardScaler()),
        ('modelo', LinearRegression())
    ])


def validacion_cruzada(pipeline: Pipeline, X: pd.DataFrame, y: pd.Series,
                       n_splits: int = 5) -> pd.DataFrame:
    kfold = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    res = cross_validate(
        pipeline, X, y, cv=kfold,
        scoring={'MAE': 'neg_mean_absolute_error',
                 'RMSE': 'neg_root_mean_squared_error',
                 'R2': 'r2'}
    )
    mae, rmse, r2 = -res['test_MAE'], -res['test_RMSE'], res['test_R2']

    return pd.DataFrame({
        'Media': [mae.mean(), rmse.mean(), r2.mean()],
        'Desviación': [mae.std(), rmse.std(), r2.std()]
    }, index=['MAE', 'RMSE', 'R2']).round(4)


def calcular_metricas(y_real: pd.Series, y_pred: np.ndarray) -> dict:
    mse = mean_squared_error(y_real, y_pred)
    return {
        'MAE': mean_absolute_error(y_real, y_pred),
        'MSE': mse,
        'RMSE': np.sqrt(mse),
        'R2': r2_score(y_real, y_pred)
    }


def obtener_coeficientes(pipeline: Pipeline, columnas: pd.Index) -> pd.DataFrame:
    return pd.DataFrame({
        'Variable': columnas,
        'Coeficiente': pipeline.named_steps['modelo'].coef_
    }).sort_values(by='Coeficiente', key=abs, ascending=False).reset_index(drop=True)


def graficar_resultados(y_real: pd.Series, y_pred: np.ndarray,
                        df_coef: pd.DataFrame, titulo: str = '') -> None:
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    fig.suptitle(titulo, fontsize=14)

    sns.scatterplot(x=y_real, y=y_pred, alpha=0.5, ax=axes[0], color='teal')
    axes[0].plot([y_real.min(), y_real.max()], [y_real.min(), y_real.max()], 'r--', lw=2)
    axes[0].set_title('Valores Reales vs. Predicciones (Test)')
    axes[0].set_xlabel('Puntaje de Productividad Real')
    axes[0].set_ylabel('Puntaje de Productividad Predicho')
    axes[0].grid(True, linestyle='--', alpha=0.6)

    colores = ['seagreen' if c > 0 else 'indianred' for c in df_coef['Coeficiente']]
    axes[1].barh(df_coef['Variable'], df_coef['Coeficiente'], color=colores)
    axes[1].invert_yaxis()
    axes[1].axvline(0, color='black', linestyle='--', linewidth=1)
    axes[1].set_title('Impacto de las Variables (Coeficientes)')
    axes[1].set_xlabel('Coeficiente (Estandarizado)')
    axes[1].grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()


# ================= FLUJO DE ENTRENAMIENTO =================
def entrenar_modelo(X_train: pd.DataFrame, X_test: pd.DataFrame,
                    y_train: pd.Series, y_test: pd.Series,
                    variables: list = None, nombre: str = 'Modelo',
                    graficar: bool = True) -> dict:

    X_tr, X_te = preparar_datos(X_train, X_test, variables)

    print(f"\n    {nombre.upper()} ({X_tr.shape[1]} variables)   ")
    print(X_tr.columns.tolist(), "\n")

    pipeline = crear_pipeline()

    metricas_cv = validacion_cruzada(pipeline, X_tr, y_train)
    print("    VALIDACIÓN CRUZADA (5 FOLDS)    ")
    display(metricas_cv)

    pipeline.fit(X_tr, y_train)
    y_pred_train = pipeline.predict(X_tr)
    y_pred_test = pipeline.predict(X_te)

    metricas = pd.DataFrame({
        'Entrenamiento': calcular_metricas(y_train, y_pred_train),
        'Test': calcular_metricas(y_test, y_pred_test)
    }).round(4)
    print("\n    MÉTRICAS    ")
    display(metricas)

    coeficientes = obtener_coeficientes(pipeline, X_tr.columns)
    print("\n    COEFICIENTES    ")
    display(coeficientes.round(4))

    if graficar:
        graficar_resultados(y_test, y_pred_test, coeficientes, titulo=nombre)

    return {
        'pipeline': pipeline,
        'metricas_cv': metricas_cv,
        'metricas': metricas,
        'coeficientes': coeficientes
    }


# ================= MENÚ =================
def elegir_opcion() -> str:
    print("¿Cómo deseas entrenar el modelo?")
    for clave, descripcion in OPCIONES.items():
        print(f"  {clave}. {descripcion}")

    opcion = input("Escribe el número de la opción: ").strip()
    while opcion not in OPCIONES:
        opcion = input("Opción no válida. Escribe 1 o 2: ").strip()
    return opcion


# ================= ORQUESTADOR =================
def main(X_train: pd.DataFrame, X_test: pd.DataFrame,
         y_train: pd.Series, y_test: pd.Series, opcion: str = None) -> dict:
    """Entrena según la opción indicada; si no se pasa ninguna, la pide con el menú."""
    if opcion is None:
        opcion = elegir_opcion()

    print(f"\nOpción elegida: {OPCIONES[opcion]}")

    if opcion == '1':
        return entrenar_modelo(X_train, X_test, y_train, y_test,
                               variables=VARIABLES_SELECCIONADAS,
                               nombre='Variables seleccionadas')

    return entrenar_modelo(X_train, X_test, y_train, y_test,
                           variables=None,
                           nombre='Todas las variables')

### **Comparación: variables seleccionadas vs. todas las variables**

Entrenamos ambas opciones para comprobar que las 10 variables descartadas no aportan información. Para usar el menú interactivo, llama a `main(X_train, X_test, y_train, y_test)` sin el parámetro `opcion`.

In [ ]:
resultado_seleccionadas = main(X_train, X_test, y_train, y_test, opcion='1')

In [ ]:
resultado_todas = main(X_train, X_test, y_train, y_test, opcion='2')

# **PREDICCIÓN SOBRE EL CONJUNTO DE PRUEBA DE KAGGLE**

Reentrenamos el modelo final (6 variables) con todo `train.csv` y predecimos `test.csv` para generar el archivo de envío.

In [ ]:
def generar_submission(df: pd.DataFrame, df_test: pd.DataFrame, variables: list,
                       ruta_salida: str = 'submission.csv') -> pd.DataFrame:
    """Entrena el pipeline con todo el conjunto de entrenamiento y guarda las predicciones de test."""
    pipeline = crear_pipeline()
    pipeline.fit(df[variables], df['puntaje_productividad'])

    submission = pd.DataFrame({
        'id': df_test['id'],
        'productivity_score': pipeline.predict(df_test[variables])
    })
    submission.to_csv(ruta_salida, index=False)
    print(f"Archivo guardado en {ruta_salida} ({len(submission)} filas)")
    return submission

submission = generar_submission(df, df_test, VARIABLES_SELECCIONADAS)
display(submission.head())

## **Conclusiones de las distribuciones**

Observo que todas las variables predictoras tienen una distribución aproximadamente uniforme, mientras que puntaje_productividad tiene forma de campana, cercana a una normal. Esto se debe a que el dataset es sintético y que la variable objetivo probablemente se construyó como una combinación lineal de unas pocas variables.

Las distribuciones no cambian mi decisión de eliminar las 10 variables irrelevantes, ya que su forma no compensa que no aportan información sobre la variable objetivo.

En las 6 variables que conservo no encuentro outliers, sesgos ni valores fuera de rango, así que no aplico transformaciones. Nivel_estres es discreta (valores enteros de 1 a 10), lo que no representa un problema para el modelo. Sí voy a escalar estas variables, porque tienen rangos distintos y eso afecta la interpretación de los coeficientes en una regresión lineal.

Entrené el modelo con todas las variables y obtuve el mismo desempeño que con las 6 seleccionadas. Además, el modelo asignó coeficientes de cero a las 10 variables que había identificado como irrelevantes. Esto confirma, de forma independiente a la correlación y la información mutua, que esas variables no aportan información, por lo que me quedo con el modelo de 6 variables por ser más simple e interpretable.

## Conclusión del ejercicio

Analicé un dataset de Kaggle para predecir la productividad de estudiantes a partir de 16 variables.

Con Pearson, Spearman e información mutua identifiqué que solo 6 variables influyen en la productividad: horas de estudio, enfoque, sueño, uso del teléfono, estrés y asistencia. Las otras 10 no tienen ninguna relación con el objetivo, y lo confirmé al entrenar con todas las variables, ya que el modelo les asignó coeficiente cero.

Entrené una regresión lineal con un Pipeline y validación cruzada, y obtuve un R² de 1.0 con un error de 0.0025, que coincide exactamente con el error de redondear a 2 decimales. Desconfié de un resultado tan perfecto y encontré la explicación: el dataset es sintético, la productividad se generó con una fórmula lineal de esas 6 variables, y mi modelo la reconstruyó.

Con este ejercicio practiqué análisis exploratorio, selección de variables, pipelines, validación cruzada y organización del código en funciones. También aprendí a verificar los resultados en lugar de aceptarlos sin más. Como el dataset no tiene ruido, no sirve para comparar modelos más complejos, así que el siguiente paso es trabajar con datos reales.